# GBT optimisation — what the tuning round actually bought

Reads the completed runs out of `mlflow.db` and `optuna.db`. **It fits nothing**, which is
deliberate: LightGBM and PyTorch each ship an OpenMP runtime, and loading both into one
process segfaults on macOS (see `src/home_credit_risk/evaluation.py`). A notebook that
only *reads* results imports neither, so this one and the NN notebook can be open at once.

Everything here is on the canonical split — `cv_seed=42`, `n_splits=5`. Runs on a
different split are not comparable and are not shown.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

# Work from the repo root, not the notebook's directory. Three things depend on it:
# `config.Settings` discovers `.env` (and API_KEY has no default, so importing
# `tracking` fails without it), and both `sqlite:///mlflow.db` and `sqlite:///optuna.db`
# are relative URIs. Guarded so re-running the cell is a no-op.
HERE = [Path.cwd(), *Path.cwd().parents]
ROOT = next(p for p in HERE if (p / "pyproject.toml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

# Imports follow the path setup above, hence the E402 suppressions -- the same pattern
# the runners in scripts/ use.
import logging  # noqa: E402

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import polars as pl  # noqa: E402

from home_credit_risk.tracking import fold_scores, run_history  # noqa: E402

# `home_credit_risk.logger` configures the ROOT logger, and APP_ENV=development sets it
# to DEBUG — which means matplotlib's font-matching chatter (hundreds of lines per
# figure) lands in every cell. Raise the floor for third-party loggers only; our own
# module's messages still come through.
for noisy in ("matplotlib", "PIL", "urllib3", "alembic", "mlflow", "asyncio"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

# The seed-noise floor measured across 5 NN seeds: any difference smaller than ~3 SE
# is not established by a single pair of runs. Drawn on every comparison below.
SEED_STD = 0.00030
SE_DIFF = SEED_STD * np.sqrt(2)
print(
    f"noise floor: std {SEED_STD:.5f}, SE of a difference {SE_DIFF:.5f}, "
    f"3 SE = {3 * SE_DIFF:.5f}"
)

## 1. The trajectory

Chronological, not sorted by score. A run that lost is evidence about the choice it
tested, so the losers stay in the picture.

In [ ]:
hist = run_history()
gbt = hist.filter(pl.col("run_name").str.starts_with("gbt-")).sort("started")

view = gbt.select(["run_name", "oof_roc_auc", "oof_pr_auc", "n_features"])
print(view)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(gbt.height)
y = gbt["oof_roc_auc"].to_numpy()
ax.plot(x, y, "o-", color="#2b6cb0", lw=2, zorder=3)
for xi, yi, name in zip(x, y, gbt["run_name"]):
    ax.annotate(
        f"{yi:.5f}",
        (xi, yi),
        textcoords="offset points",
        xytext=(0, 9),
        ha="center",
        fontsize=8,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(gbt["run_name"], rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("OOF ROC-AUC")
    ax.set_title("GBT tuning trajectory (chronological)")
    ax.margins(y=0.18)
plt.tight_layout()

## 2. The budget probe — was the tree cap binding?

The original baseline capped at 1000 trees and **early stopping never fired on any fold**:
the cap, not convergence, ended every fit, so the configuration's ceiling was unknown. The
probe re-ran with a cap of 6000 to find where it actually converges.

`auc_loss_at_0.7x` / `auc_loss_at_1.3x` measure how much AUC is given up at 70% and 130%
of the peak iteration — i.e. **how flat the optimum is**. A flat optimum means the exact
tree count barely matters, which is what licenses fixing it for the final re-fit.

In [ ]:
budget = gbt.filter(pl.col("run_name") == "gbt-budget-6000")
cols = [
    "peak_iteration",
    "peak_auc",
    "auc_loss_at_0.7x",
    "auc_loss_at_1.3x",
    "median_best_iteration",
    "mean_best_iteration",
]
present = [c for c in cols if c in budget.columns]
if budget.height:
    row = budget.select(present).row(0, named=True)
    for k, v in row.items():
        print(f"  {k:24s} {v:,.5f}" if v is not None else f"  {k:24s} —")
    peak = row.get("peak_iteration")
    loss_lo, loss_hi = row.get("auc_loss_at_0.7x"), row.get("auc_loss_at_1.3x")
    if peak and loss_lo is not None:
        print(f"\n  Peak at ~{peak:,.0f} trees. 30% short costs {loss_lo:.5f} AUC;")
        print(f"  30% long costs {loss_hi:.5f}. Both are within the {3 * SE_DIFF:.5f}")
        print("  noise floor -> the optimum is flat, so fixing the count is safe.")

## 3. The Optuna search

Four regularisation parameters searched **jointly** at `lr=0.05` (cheaper trials), with
the incumbent enqueued as trial 0 so the search always has a real baseline to beat.

In [ ]:
import optuna  # noqa: E402

study = optuna.load_study(study_name="gbt-screening", storage="sqlite:///optuna.db")
trials = pl.DataFrame(
    [
        {"trial": t.number, "state": t.state.name, "value": t.value, **t.params}
        for t in study.trials
    ]
)
print(trials)
print(f"\nbest: trial {study.best_trial.number}  value {study.best_value:.5f}")
print(f"params: {study.best_params}")

In [ ]:
done = trials.filter(pl.col("state") == "COMPLETE")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.plot(done["trial"], done["value"], "o-", color="#2b6cb0", zorder=3)
running = np.maximum.accumulate(done["value"].to_numpy())
ax.step(
    done["trial"], running, where="post", color="#c05621", lw=2, label="running best"
)
ax.set_xlabel("trial")
ax.set_ylabel("mean fold ROC-AUC")
ax.set_title("Search history (lr=0.05)")
ax.legend(fontsize=8)

ax = axes[1]
ax.scatter(
    done["num_leaves"],
    done["value"],
    s=90,
    c=done["min_child_samples"],
    cmap="viridis",
    zorder=3,
)
for r in done.iter_rows(named=True):
    ax.annotate(
        str(r["trial"]),
        (r["num_leaves"], r["value"]),
        textcoords="offset points",
        xytext=(7, 0),
        fontsize=8,
    )
    ax.set_xlabel("num_leaves")
    ax.set_ylabel("mean fold ROC-AUC")
    ax.set_title("colour = min_child_samples")
plt.tight_layout()

**Three things in that table are traps, and all three are worth reading before the plot.**

1. **The pruned trial shows the highest `value` in the column — ignore it.** A pruned
   trial's value is whatever it had reported when the pruner killed it, which here is a
   *single fold*, not a 5-fold mean. Fold 2 is the easiest fold in this split (visible in
   §4), so an unlucky pruned trial can look like the winner. Only `COMPLETE` rows are
   comparable, which is why the plot below filters to them.
2. **Trial 4 reproduces trial 1 exactly** — same parameters, same score to every decimal.
   That is a resumed-study artifact: a seeded TPE sampler restarts its random sequence
   from the beginning when the study is reloaded, so it re-proposes the same points. The
   fix (offsetting the sampler seed by the existing trial count) is in
   `scripts/tune_gbt.py`; the duplicate is left in the study as a record.
3. **Trial 5 failed** after being given the same parameters as trial 2 — it hit the
   wall-clock guard, not a modelling error.

> **On what this search does and does not establish.** A handful of completed trials is
> enough to have found a configuration that beats the incumbent. It is **not** enough to
> support a parameter-importance ranking — with this few points an importance plot would
> mostly be describing the sampler's startup randomness, which is why none is shown. The
> honest summary is "the search found a better point", not "the search told us what
> matters".

## 4. Fold-level detail — is the improvement real?

Every run uses the identical split, so fold *k* of one run pairs with fold *k* of
another. That pairing removes fold difficulty from the comparison and is much more
sensitive than comparing two OOF scalars.

In [ ]:
pair = ["gbt-fixes-and-ratios", "gbt-final"]
fs = fold_scores(pair)

if fs.height:
    wide = fs.pivot(values="score", index="fold", on="run_name").sort("fold")
    base, tuned = pair[0], pair[1]
    wide = wide.with_columns((pl.col(tuned) - pl.col(base)).alias("delta"))
    print(wide)

    wins = int((wide["delta"] > 0).sum())
    print(f"\ntuned config wins on {wins}/{wide.height} folds")
    print(f"mean delta {wide['delta'].mean():+.5f}  (3 SE = {3 * SE_DIFF:.5f})")

    fig, ax = plt.subplots(figsize=(7, 4))
    w = 0.36
    f = wide["fold"].to_numpy()
    ax.bar(f - w / 2, wide[base], w, label=base, color="#a0aec0")
    ax.bar(f + w / 2, wide[tuned], w, label=tuned, color="#2b6cb0")
    ax.set_ylim(min(wide[base].min(), wide[tuned].min()) - 0.004, None)
    ax.set_xticks(f)
    ax.set_xlabel("fold")
    ax.set_ylabel("ROC-AUC")
    ax.set_title("Paired fold comparison")
    ax.legend(fontsize=8)
    plt.tight_layout()

## 5. What the model actually uses

Gain importance from the final re-fit. Gain is **null-native** — it measures the loss
reduction a feature delivered at its split points, so a feature that is 60% missing but
decisive when present ranks on its usefulness, not its coverage.

In [ ]:
imp_path = Path("~/code/datasets/gbt_gain_importance.csv").expanduser()
if imp_path.exists():
    imp = pl.read_csv(imp_path)
    gain_col = "gain_norm" if "gain_norm" in imp.columns else imp.columns[1]
    top = imp.sort(gain_col, descending=True).head(25)

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.barh(top["feature"][::-1], top[gain_col][::-1], color="#2b6cb0")
    ax.set_xlabel(gain_col)
    ax.set_title("Top 25 features by gain")
    plt.tight_layout()

    share = top[gain_col].sum() / imp[gain_col].sum()
    print(f"top 25 of {imp.height} features carry {share:.1%} of total gain")
else:
    print(f"not found: {imp_path} — regenerate with scripts/tune_gbt.py final")

## 6. Where it landed

| | OOF ROC-AUC |
|---|---|
| Baseline (pre-tuning) | 0.79253 |
| Budget probe, 6000 cap | 0.79306 |
| Tuned, transferred to `lr=0.02` | **0.79488** |
| Final re-fit, fixed trees, no early stopping | **0.79478** |

The **final** row is the quotable number: no early stopping means no fold-validation
signal reached the fitted model, so nothing about the score is optimistic.

Tuning bought roughly **+0.0022** over the pre-tuning baseline. Worth having, and small
next to what the feature work and the model class contributed — the same tree beats a
properly-built logistic model on identical features by **+0.017**.